# Teoría del EDA

Los ejemplos son los de clase

## Qué es

EDA son las siglas de Exploratory Data Analysis, en español análisis exploratorio de datos. Es lo primero que se hace cuando te llega una tabla nueva, antes de sacar conclusiones, de hacer gráficos para un informe o de entrenar un modelo.

Consiste en mirar los datos con calma y desconfiando un poco. Qué hay, qué falta, qué está mal y qué cosas llaman la atención.

## Por qué es importante

Los datos reales casi nunca vienen limpios. Siempre falta algo, hay filas repetidas o alguien se equivocó al teclear. Si no lo buscas, los cálculos salen mal y nadie te avisa.

En clase lo vimos con los hoteles. Con los datos tal cual llegaron, se cancelaba el 37 % de las reservas. Después de quitar las filas repetidas, el 28 %. La pregunta era la misma y la respuesta cambió mucho.

También sirve para saber si tu pregunta se puede responder con esos datos. Los hoteles tenían reservas de julio de 2015 a agosto de 2017, así que solo 2016 estaba completo. Comparar años con esa tabla habría dado una conclusión falsa.

Y es el paso que va antes de cualquier modelo de machine learning. Un modelo aprende de los datos que le das, con sus errores incluidos.

## Dónde encaja

Un proyecto de datos suele seguir estos pasos:

1. Hacerse una pregunta.
2. Conseguir los datos, de un CSV, de una API o de una base de datos.
3. Limpiarlos y explorarlos. Esto es el EDA.
4. Hacer un modelo, si la pregunta lo necesita.
5. Contar lo que has encontrado.

En la mayoría de proyectos se pasa más tiempo en el paso 3 que en el 4.

## Las 9 preguntas

Hacer un EDA es ir haciéndole preguntas a la tabla, más o menos siempre en este orden:

1. ¿Qué tengo? Cuántas filas y columnas hay, qué es cada fila y de qué tipo es cada columna.
2. ¿Qué me falta?
3. ¿Hay filas repetidas?
4. ¿Cómo son los números?
5. ¿Hay algo raro?
6. ¿Qué hay más?
7. ¿Cambia algo de un grupo a otro?
8. ¿Van juntas dos columnas?
9. ¿Qué he aprendido?

Las cinco primeras sirven sobre todo para conocer y limpiar la tabla. Con las otras cuatro empiezan a salir las ideas.

Una buena pregunta es concreta y se responde con una o dos líneas de código. "¿Qué hay en los datos?" no lleva a ningún sitio. "¿Se cancela más en el hotel de ciudad?" sí, porque enseguida te lleva a preguntarte por qué.

## Los huecos

Un hueco es un dato que no está, y en pandas sale como NaN. En `info()` se ven en la columna Non-Null Count, que dice cuántos valores hay en cada columna. Si hay menos valores que filas, faltan datos.

Lo primero es mirarlo en porcentaje. 488 huecos parecen muchos, pero en una tabla de 119.390 filas son el 0,4 %.

Después, con cada columna se toma una de estas tres decisiones:

| Decisión | Cuándo | Ejemplo de clase |
|---|---|---|
| Quitar la columna | Le falta casi todo | La empresa en los hoteles, vacía en el 94 % |
| Quitar las filas | Faltan pocas, o el dato no se puede inventar | Las reservas sin país y los pisos sin precio |
| Rellenar | Sabes qué significa el hueco, o un valor normal como la mediana no cambia nada | Sin agente es una reserva directa, así que 0. Sin licencia, "sin licencia" |

Rellenar es inventarse un poco el dato, así que se hace solo cuando tiene sentido y luego se dice.

Hay que tener cuidado también con los huecos disfrazados. A veces el dato falta pero en vez de NaN pone "Undefined", "-", 0 o 999, y `isna` no los cuenta. En los hoteles, la comida tenía 1.165 "Undefined". Por eso conviene mirar con `value_counts` lo que hay en las columnas de texto.

## Los duplicados

Una fila repetida cuenta dos veces en todo, en los totales, en las medias y en los porcentajes. Se cuentan con `duplicated().sum()` y se quitan con `drop_duplicates()`.

Antes de borrarlas hay que pensar si de verdad son la misma fila. Si la tabla tiene un identificador, como un número de reserva o el id de un anuncio, fíate de él. En Airbnb no había ningún id repetido. Sin el id salían 216 filas iguales, pero eran anuncios distintos con los mismos datos, así que no se borran.

Los hoteles no tenían número de reserva. Ahí las quitamos, pero hay que decirlo al contar los resultados.

## Media y mediana

La media es sumar todo y dividir entre cuántos hay. La mediana es el valor que queda en medio cuando los ordenas.

Se entiende mejor con un ejemplo. Si cobras cinco cañas a 2, 2, 2, 3 y 50 euros porque te equivocaste en una, la media sale 11,8 € y la mediana 2 €. Lo que cuesta una caña lo dice mejor la mediana.

Cuando la media y la mediana se parecen, los datos están equilibrados. Cuando la media es mucho más alta, hay unos pocos valores enormes que tiran de ella hacia arriba, y es mejor usar la mediana. En Airbnb la media del precio era 208,7 € y la mediana 151 €.

El histograma parte los valores en tramos y cuenta cuántos caen en cada uno, y así se ve la forma de los datos. Si sale casi todo en una barra, hay valores extremos aplastando el gráfico.

Y no todo lo que es número es una cantidad. `describe` saca la media del id o del número de agente, pero esa media no significa nada.

## Los valores raros

Un valor atípico es uno que se sale mucho de lo normal, como Jabba entre los personajes de Star Wars. Cuando aparece uno hay que decidir qué es.

- Si es imposible, es un error, como un precio negativo o 10 bebés en una reserva. Se quita.
- Si podría ser las dos cosas, como un piso a 15.000 € la noche, se pregunta a quien conoce los datos. Mientras tanto, se decide algo con sentido y se dice.
- Si es raro pero posible, como una reserva de 55 adultos o el propio Jabba, se deja. Para que no tuerza las cuentas, mejor usar la mediana.

Un valor raro no se borra porque moleste, solo si es un error. Y el corte que pongas, por ejemplo quedarte con los pisos de menos de 1.000 €, es una decisión tuya que tienes que explicar.

## Categorías y grupos

`value_counts` cuenta cuántas filas hay de cada categoría. Con `normalize=True` da la proporción, y multiplicando por 100, el porcentaje.

`groupby` sirve para comparar grupos. Calcula la media o la mediana de una columna para cada valor de otra, por ejemplo el precio mediano de cada distrito. De aquí salen las ideas, sobre todo cuando algo no cuadra. Las reservas que no se podían devolver se cancelaban el 95 % de las veces, justo al revés de lo que esperarías. El EDA no explica por qué, pero te dice qué hay que preguntar.

Antes de fiarte de una media, mira cuántas filas tiene el grupo. Ese 95 % salía de 971 reservas, y otro de los grupos tenía solo 84. Con tan pocas filas, cualquier conclusión se cae enseguida.

## Relaciones entre columnas

En un gráfico de puntos cada fila es un punto. Si la nube sube, las dos columnas van juntas. Si es una mancha sin forma, no tienen que ver.

La correlación (`corr`) lo resume en un número entre -1 y 1. Cerca de 1, cuando una sube la otra también. Cerca de -1, cuando una sube la otra baja. Y cerca de 0, no hay relación.

Que dos cosas vayan juntas no quiere decir que una provoque la otra. En verano se venden más helados y hay más ahogados, y no es por los helados, es porque hace calor.

## Contar lo que has encontrado

Cada hallazgo tiene que llevar un número y algo de contexto. "Hay cancelaciones" no dice nada. "Se cancela el 28 % de las reservas, más en el hotel de ciudad (31 %) que en el de playa (25 %)" sí.

Cuenta también qué has quitado y por qué. Es lo que hace que los demás se fíen de tus números.

## Pandas: qué usar y cuándo

Esto es lo que más vas a usar de pandas en un EDA, agrupado por para qué sirve. En los ejemplos, `tabla` es tu tabla (como `hoteles` o `pisos`) y "a" y "b" son nombres de columnas.

Antes de nada, lo que más falla. Casi nada de pandas cambia la tabla por sí solo. `tabla.dropna()` te enseña cómo quedaría, pero la tabla sigue igual. Para que cambie hay que guardarlo, con `tabla = tabla.dropna()`. Pasa lo mismo con `drop`, `fillna`, `drop_duplicates`, `replace` y los filtros.

### Mirar la tabla

- `tabla.head()` enseña las 5 primeras filas y `tabla.tail()` las 5 últimas. Con `head(10)` salen 10.
- `tabla.shape` da las filas y las columnas, por ejemplo (119390, 32). No lleva paréntesis porque no hace nada, es un dato que la tabla ya tiene.
- `len(tabla)` da solo el número de filas. Es lo que se usa para contar, por ejemplo cuántas filas quedan después de un filtro.
- `tabla.columns` da los nombres de las columnas. Si le das una lista, `tabla.columns = ["x", "y"]`, los cambias todos de golpe y en el mismo orden.
- `tabla.info()` dice de cada columna de qué tipo es y cuántos valores tiene (Non-Null Count). Es lo primero que se mira.
- `tabla.describe()` hace el resumen de las columnas con números, con cuántos valores hay, la media, el mínimo, el máximo y los cuartiles.

### Los huecos: isna, dropna y fillna

Las tres tienen que ver con los huecos, pero hacen cosas muy distintas.

- `isna()` solo pregunta. No cambia nada, pone True donde falta el dato y False donde está. Con `tabla.isna().sum()` cuentas los huecos de cada columna.
- `dropna()` quita filas. Sin nada entre los paréntesis, quita cualquier fila que tenga un hueco en cualquier columna, y eso se puede llevar por delante muchas filas buenas. En los hoteles habríamos perdido unas 16.000 reservas solo porque no tenían agente.
- `dropna(subset=["a"])` solo mira las columnas que le pongas. Es la forma normal de usarlo.
- `fillna(0)` rellena los huecos con lo que le digas. Se suele usar con una sola columna, `tabla["a"] = tabla["a"].fillna(0)`.
- `replace("Undefined", "SC")` cambia un valor por otro. Sirve para los huecos disfrazados, que no son NaN y por eso `fillna` no los ve.
- `value_counts(dropna=False)` cuenta también los huecos, que normalmente no salen en la cuenta.

### Quitar cosas: drop, dropna y drop_duplicates

- `drop(columns=["a", "b"])` quita las columnas que le digas por su nombre, tengan lo que tengan.
- `drop(index=[0, 5])` quita filas por su número. Se usa poco.
- `dropna(...)` quita filas porque tienen huecos.
- `drop_duplicates()` quita las filas repetidas y deja la primera de cada grupo. Con `subset=["id"]` cuenta como repetidas las filas que tienen el mismo id, aunque el resto cambie.
- `duplicated()` no quita nada, marca con True las filas repetidas. Con `.sum()` las cuentas y con `tabla[tabla.duplicated()]` las ves.

Con `drop` decides tú qué se va. Con `dropna` y `drop_duplicates` lo decide pandas, siguiendo una regla.

### Elegir columnas y filtrar filas

- `tabla["a"]` es una columna sola.
- `tabla[["a", "b"]]` son varias columnas, y el resultado sigue siendo una tabla. El corchete de dentro es la lista de nombres.
- `tabla[tabla["a"] > 0]` deja las filas que cumplen la condición.
- `tabla[(tabla["a"] > 0) & (tabla["b"] == "x")]` deja las que cumplen las dos. Con `|` en vez de `&`, las que cumplen al menos una. Cada condición va entre paréntesis.
- `tabla.sort_values("a", ascending=False)` ordena de mayor a menor. Sin `ascending=False`, de menor a mayor.

### Contar y resumir

- `tabla["a"].value_counts()` cuenta cuántas veces sale cada valor. Con `normalize=True` da la proporción, y multiplicando por 100, el porcentaje.
- `tabla["a"].unique()` da los valores distintos que hay, y `nunique()` cuántos son. Por ejemplo, cuántos distritos distintos hay.
- `mean()`, `median()`, `min()`, `max()` y `sum()` hacen lo que dice su nombre.
- `count()` cuenta los valores que no están vacíos. Por eso puede salir menos que `len(tabla)`.
- `round(1)` deja un decimal.
- `tabla.groupby("a")["b"].median()` saca la mediana de b para cada valor de a. Se lee en orden. Agrupa por a, coge la columna b y calcula la mediana de cada grupo.
- `tabla["a"].corr(tabla["b"])` da la correlación entre dos columnas.

### Gráficos

Todos se hacen con `plot`, y lo que cambia es el `kind`.

- `kind="hist"` es el histograma, para ver cómo se reparten los valores de una columna de números.
- `kind="box"` es el diagrama de caja, para encontrar valores extremos.
- `kind="bar"` son barras, para comparar categorías. Suele ir detrás de un `value_counts()` o de un `groupby`.
- `kind="scatter"` es el gráfico de puntos, para ver si dos columnas de números van juntas. Necesita `x` e `y`.

### Leer y guardar

- `pd.read_csv("datos/archivo.csv")` lee un CSV y lo convierte en una tabla.
- `tabla.to_csv("archivo.csv", index=False)` la guarda en un CSV. Con `index=False` no se guarda el número de fila como si fuera una columna más.